# TerraClimate monthly regional tables


Decode the 150 downloaded V1.1 NetCDF files without resampling. Produce native-cell monthly values and area-weighted regional monthly summaries. Each summary averages valid cell centers inside the original box with cosine-latitude weights; it is an approximation to equal-area weighting, not partial-cell polygon integration. Coverage is reported over all selected grid centers, including sea cells that have no land climate values.

Temperature minima/maxima are monthly averages, precipitation is a monthly total, wind and radiation are monthly fields. No daily extremes, annual totals, climatological normals, or lunar temperature equivalences are inferred.


Run cells in order manually. Raw inputs are read-only. Each processing run writes to a new directory under `data/processed/`; only a `provenance.json` with `status: complete` indicates successful completion. See the [notebook index](../README.md) for environment setup and dependencies.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "outline.md").is_file() and (p / "data/raw").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the project root or a folder under notebooks")
sys.path.insert(0, str(ROOT / "notebooks"))
print("Project:", ROOT)


In [ ]:
import json
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from scipy.io import netcdf_file
from processing import AREAS, Run, decode_packed, sha256
sys.path.insert(0, str(ROOT / "download scripts"))
from download_terraclimate import validate_netcdf

SELECTED_AREAS = list(AREAS)
EXPECTED_YEARS = list(range(2015, 2025))
VARIABLES = ["tmin", "tmax", "ppt", "ws", "srad"]
base = ROOT / "data/raw/earth/terraclimate"
plans = sorted((base / "manifests").glob("*_run.json"))
if not plans:
    raise FileNotFoundError("No TerraClimate download run")
expected = {(a, y, v) for a in SELECTED_AREAS for y in EXPECTED_YEARS for v in VARIABLES}
compatible = []
for candidate in plans:
    record = json.loads(candidate.read_text())
    selected = [j for j in record["jobs"] if j["area"] in SELECTED_AREAS]
    if record["status"] == "complete" and {(j["area"], j["year"], j["variable"]) for j in selected} == expected and len(selected) == len(expected):
        compatible.append((candidate, selected))
if not compatible:
    raise ValueError("No complete download manifest matches the requested areas, years and variables")
manifest_path, jobs = compatible[-1]
assert {(j["area"], j["year"], j["variable"]) for j in jobs} == expected
assert len(jobs) == len(expected)
print("Files:", len(jobs))


In [ ]:
with Run(ROOT, "earth_climate", {"areas": SELECTED_AREAS, "years": EXPECTED_YEARS,
         "variables": VARIABLES, "weighting": "cos(latitude), centers inside bbox"}, "preprocessing/03_earth_climate.ipynb") as run:
    run.input(manifest_path)
    run.input(ROOT / "download scripts/download_terraclimate.py")
    summaries = []
    for job in jobs:
        path = base / job["path"]
        receipt_path = path.with_suffix(".nc.json")
        receipt = json.loads(receipt_path.read_text())
        assert sha256(path) == receipt["sha256"] and receipt["source"] == job["source"]
        assert tuple(job["bounds"]) == tuple(AREAS[job["area"]]["bbox"])
        validate_netcdf(path, job["variable"], job["year"], tuple(job["bounds"]))
        run.input(path)
        run.input(receipt_path)
        with netcdf_file(str(path), "r", mmap=False, maskandscale=False) as dataset:
            variable = dataset.variables[job["variable"]]
            units = variable.units.decode() if isinstance(variable.units, bytes) else str(variable.units)
            data = decode_packed(variable[:], [getattr(variable, "_FillValue", -32768),
                                  getattr(variable, "missing_value", -32768)],
                                  float(getattr(variable, "scale_factor", 1)), float(getattr(variable, "add_offset", 0)))
            lon, lat = np.meshgrid(dataset.variables["lon"][:], dataset.variables["lat"][:])
            west, south, east, north = job["bounds"]
            inside = (lon >= west) & (lon < east) & (lat >= south) & (lat < north)
            weights = np.cos(np.deg2rad(lat))
            dates = [datetime(1900, 1, 1) + timedelta(days=float(t)) for t in dataset.variables["time"][:]]
            records = []
            for index, date in enumerate(dates):
                values = data[index]
                valid = inside & np.isfinite(values)
                if not valid.any():
                    raise ValueError(f"No valid climate cells: {path.name}, {date}")
                records.append(pd.DataFrame({"date": date.date().isoformat(), "longitude": lon[inside],
                    "latitude": lat[inside], "value": values[inside], "valid": valid[inside],
                    "variable": job["variable"], "units": units}))
                summaries.append({"area": job["area"], "date": date.date().isoformat(), "variable": job["variable"],
                    "units": units, "spatial_weighted_mean": float(np.average(values[valid], weights=weights[valid])),
                    "spatial_min_of_monthly_field": float(values[valid].min()),
                    "spatial_max_of_monthly_field": float(values[valid].max()),
                    "valid_cells": int(valid.sum()), "selected_cells": int(inside.sum()),
                    "weighted_coverage": float(weights[valid].sum() / weights[inside].sum())})
            output = run.path(f"earth/{job['area']}/native_monthly/{job['variable']}_{job['year']}.csv.gz")
            pd.concat(records, ignore_index=True).to_csv(output, index=False, compression={"method": "gzip", "mtime": 0})
    summary_frame = pd.DataFrame(summaries)
    summary_frame.to_csv(run.path("regional_monthly_summary.csv"), index=False)
    assert len(summary_frame) == len(jobs) * 12
    run.record["summary_rows"] = len(summary_frame)
print("Completed:", run.directory)
print(summary_frame.groupby(["area", "variable"])["weighted_coverage"].agg(["min", "max"]))
